# Lab 06 · Data skew

**Part A** (about 50 min): build a skewed join, find the slow task, and fix it four ways; then see what `repartition` and `coalesce` do.<br>
**Part B** (about 70 min): 12 problems: latest rows, duplicates, sessions and streaks.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>4.1</b> What data skew is · <b>4.2</b> Finding skew in the Spark UI · <b>4.3</b> Why skew is dangerous · <b>4.4</b> Ways to fix skew · <b>4.5</b> Salting · <b>4.6</b> AQE skew join · <b>4.7</b> Does filtering help? · <b>4.8</b> repartition() and skew · <b>4.9</b> Skew on the read side · <b>4.10</b> repartition() vs. coalesce() · <b>4.11</b> Iterative salting · <b>6.2</b> When to use repartition() or coalesce()</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`, `Window` is `pyspark.sql.Window`.
* **Part A**: run the code under the question (or add a line), look at the output or the Spark UI, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Web UI: Stages tab](https://spark.apache.org/docs/latest/web-ui.html#stages-tab): summary metrics (min, median, max) of a stage's tasks
* [AQE: optimizing skew join](https://spark.apache.org/docs/latest/sql-performance-tuning.html#optimizing-skew-join)
* [Coalescing post-shuffle partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#coalescing-post-shuffle-partitions)
* [DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html) and [DataFrame.coalesce](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html)
* [Window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html): `row_number`, `lag`, running sums
* [DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `unix_timestamp`, `sequence`, `explode`, `date_sub`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also makes:
* **skewed_orders** (Part A): 4,000,000 generated orders (*order_id*, *customer_id*, *amount*). Customer 1 places 60% of them; the rest are spread over 100,000 customers.
* **many_customers** (Part A): 100,001 customers (*customer_id*, *region*).
* **events** (Part B): website visits: *customer_id*, *event_time* (a timestamp), *page*. Two rows were loaded twice.

## Setup
Run this cell once. It starts Spark, saves the three shop tables as Parquet files in `data/shop/`, and defines *skewed_orders*, *many_customers*, *events* and the helpers `noop`, `task_report` and `partition_sizes`.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab06")
         .config("spark.sql.adaptive.enabled", "false")        # AQE starts off; question A3 turns it on
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "16")         # 16 join tasks make the skew easy to see
         .config("spark.sql.autoBroadcastJoinThreshold", "-1")  # no broadcast joins until question A6
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def noop(df):
    """Runs the whole query but keeps no result (Spark's "noop" sink)."""
    df.write.format("noop").mode("overwrite").save()

import json, urllib.request

def spark_api(path):
    """Reads the Spark UI's REST API, e.g. spark_api("/stages")."""
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

def task_report():
    """Rows and time of the tasks in the biggest stage of the last finished job: median and maximum."""
    job = max(spark_api("/jobs?status=succeeded"), key=lambda j: j["jobId"])
    stages = [spark_api(f"/stages/{s}")[0] for s in job["stageIds"]]
    stage = max((s for s in stages if s["status"] == "COMPLETE"), key=lambda s: s["numTasks"])
    tasks = spark_api(f"/stages/{stage['stageId']}/{stage['attemptId']}/taskList?length=10000")
    rows = sorted(t["taskMetrics"]["shuffleReadMetrics"]["recordsRead"] for t in tasks)
    ms = sorted(t["duration"] for t in tasks)
    return (f"{len(rows)} tasks | rows per task: median {rows[len(rows) // 2]:,}, max {rows[-1]:,} "
            f"| time: median {ms[len(ms) // 2]} ms, max {ms[-1]} ms")

def partition_sizes(df):
    """Rows in each partition of df, biggest first."""
    return [r["count"] for r in df.groupBy(F.spark_partition_id()).count().orderBy(F.desc("count")).collect()]

def pick(salt, n):
    """A number from 0 to n-1 computed from the row id: looks random, but is the same on every machine."""
    return F.pmod(F.xxhash64("id", F.lit(salt)), F.lit(n))

# Part A: 4,000,000 generated orders where customer 1 (a big reseller) places 60% of all orders
skewed_orders = spark.range(4_000_000).select(
    F.col("id").alias("order_id"),
    F.when(pick(1, 10) < 6, F.lit(1)).otherwise(pick(2, 100_000) + 2).alias("customer_id"),
    pick(3, 1000).cast("double").alias("amount"))
many_customers = spark.range(1, 100_002).select(F.col("id").alias("customer_id"), pick(4, 50).alias("region"))

# Part B: website events (two rows were loaded twice)
events = spark.createDataFrame([
    # customer_id, event_time, page
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
orders.show()
events.show()
skewed_orders.show(3)

## Part A · Find the skew, then fix it

### Finding skew <span style="color:#d93025">(4.1, 4.2, 4.3)</span>

**A1. Which customer has the most orders? Then run the plain join of skewed_orders with many_customers and look at `task_report()` and at the join stage in the Spark UI (Stages tab → Summary Metrics): how does the biggest task compare with the median one?**

In [ ]:
# My answer:
skewed_orders.groupBy("customer_id").count().orderBy(F.desc("count")).show(3)
plain_join = skewed_orders.join(many_customers, "customer_id")
noop(plain_join)
print(task_report())

### Four ways to fix it <span style="color:#d93025">(4.4, 4.5, 4.6, 4.7, 4.11)</span>

**A2. Fix it with **salting**: give every order a salt from 0 to 15, copy every customer 16 times (once per salt), and join on customer_id and salt. Compare the biggest and the median task with A1.**

In [ ]:
# My answer:
SALTS = 16
salted_orders = skewed_orders.withColumn("salt", F.pmod(F.xxhash64("order_id"), F.lit(SALTS)))      # 0..15 per order
salted_customers = many_customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(SALTS - 1))))   # 16 copies
salted_join = salted_orders.join(salted_customers, ["customer_id", "salt"])
noop(salted_join)
print(task_report())

**A3. Salt only the hot key (a selective, or iterative, salting): customer 1 gets salts 0 to 15, everybody else salt 0. How many customer rows does that need, and how balanced are the tasks?**

In [ ]:
# My answer:
hot = F.col("customer_id") == 1
hot_orders = skewed_orders.withColumn("salt", F.when(hot, F.pmod(F.xxhash64("order_id"), F.lit(SALTS))).otherwise(0))
hot_customers = (many_customers
                 .withColumn("salt", F.when(hot, F.sequence(F.lit(0), F.lit(SALTS - 1))).otherwise(F.array(F.lit(0))))
                 .withColumn("salt", F.explode("salt")))
print(hot_customers.count(), "customer rows instead of", many_customers.count() * SALTS)
noop(hot_orders.join(hot_customers, ["customer_id", "salt"]))
print(task_report())

**A4. Let **AQE** handle it: turn AQE and its skew join on (cell below, with lower limits because our data is small) and collect the join. What does the final plan say about the join?**

In [ ]:
# My answer:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "4MB")   # our data is small,
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "4MB")               # so lower the limits
by_region = skewed_orders.join(many_customers, "customer_id").groupBy("region").agg(F.sum("amount"))
by_region.collect()

**A5. Two more fixes: **broadcast** the small side, or **filter** the hot key out if you don't need it (for example a test account). Time the broadcast join, and look at the task report of the filtered join.**

In [ ]:
# My answer:
start = time.time()
noop(skewed_orders.join(F.broadcast(many_customers), "customer_id"))
print(f"broadcast join: {time.time() - start:.2f} s")

### repartition, coalesce and the read side <span style="color:#d93025">(4.8, 4.9, 4.10, 6.2)</span>

**A6. What do `repartition(8)` and `repartition(8, "customer_id")` do to the partition sizes of skewed_orders? Then add `.coalesce(2)` to each: what changes?**

In [ ]:
# My answer:
print("repartition(8):          ", partition_sizes(skewed_orders.repartition(8))[:3])
print("repartition(8, customer): ", partition_sizes(skewed_orders.repartition(8, "customer_id"))[:3])

**A7. Can skew happen when reading? Write skewed_orders as one gzip CSV file and read it back: how many read partitions? Fix the next steps with `repartition(8)`.**

In [ ]:
# My answer:
skewed_orders.coalesce(1).write.mode("overwrite").option("header", True).option("compression", "gzip").csv("data/shop/orders_gz")
gz = spark.read.option("header", True).csv("data/shop/orders_gz")
print(gz.rdd.getNumPartitions(), "read partition(s)")

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Latest rows, duplicates, sessions and streaks
12 problems on *orders* and *events*. They all use window functions; break ties exactly as the question says.

**B1. Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger order_id). Leave out the guest order.**

Expected output:

```
+-----------+--------+----------+---------+
|customer_id|order_id|order_date|   status|
+-----------+--------+----------+---------+
|          1|    1015|2024-03-25|  shipped|
|          2|    1013|2024-03-18|delivered|
|          3|    1016|2024-03-28|delivered|
|          4|    1010|2024-03-08|cancelled|
|          5|    1014|2024-03-20|delivered|
|          6|    1012|2024-03-12|  shipped|
+-----------+--------+----------+---------+
```

<details><summary>Hint</summary>

`row_number` over a window ordered by date descending; keep number 1.

</details>

**B2. Show each customer's second most recent order: customer_id, order_id and order_date. Customers with only one order drop out.**

Expected output:

```
+-----------+--------+----------+
|customer_id|order_id|order_date|
+-----------+--------+----------+
|          1|    1007|2024-02-20|
|          2|    1005|2024-02-05|
|          3|    1009|2024-03-05|
|          4|    1006|2024-02-15|
|          5|    1008|2024-03-02|
+-----------+--------+----------+
```

<details><summary>Hint</summary>

The same window as B1; keep number 2.

</details>

**B3. Which events are exact duplicates? Show customer_id, event_time, page and copies (how many times the row appears), for the rows that appear more than once.**

Expected output:

```
+-----------+-------------------+-------+------+
|customer_id|         event_time|   page|copies|
+-----------+-------------------+-------+------+
|          1|2024-03-01 09:10:00|product|     2|
|          2|2024-03-05 18:00:00|   home|     2|
+-----------+-------------------+-------+------+
```

<details><summary>Hint</summary>

Group by all columns and count.

</details>

**B4. Remove the exact duplicates, then count the events of each customer. Show customer_id and events.**

Expected output:

```
+-----------+------+
|customer_id|events|
+-----------+------+
|          1|     6|
|          2|     5|
|          3|     3|
+-----------+------+
```

<details><summary>Hint</summary>

`dropDuplicates()` with no arguments compares whole rows.

</details>

**B5. Show the first visit of each page by each customer: customer_id, page and first_visit, sorted by customer_id, then first_visit.**

Expected output:

```
+-----------+-------+-------------------+
|customer_id|   page|        first_visit|
+-----------+-------+-------------------+
|          1|   home|2024-03-01 09:00:00|
|          1|product|2024-03-01 09:10:00|
|          1|   cart|2024-03-02 10:00:00|
|          2|   home|2024-03-01 20:00:00|
|          2|product|2024-03-01 20:20:00|
|          2|   cart|2024-03-01 20:45:00|
|          3|product|2024-03-02 12:00:00|
|          3|   cart|2024-03-04 12:00:00|
+-----------+-------+-------------------+
```

<details><summary>Hint</summary>

The earliest time is the minimum.

</details>

**B6. For customer 1 (duplicates removed), show each event_time and the minutes since the previous event (`minutes_since_prev`, empty for the first one).**

Expected output:

```
+-------------------+------------------+
|         event_time|minutes_since_prev|
+-------------------+------------------+
|2024-03-01 09:00:00|              NULL|
|2024-03-01 09:10:00|                10|
|2024-03-01 11:00:00|               110|
|2024-03-02 10:00:00|              1380|
|2024-03-03 10:30:00|              1470|
|2024-03-05 08:00:00|              2730|
+-------------------+------------------+
```

<details><summary>Hint</summary>

`F.lag` for the previous time; `F.unix_timestamp` turns a timestamp into seconds.

</details>

**B7. Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than 30 minutes passed since the customer's previous event. Show customer_id, event_time, page and session (1, 2, ... per customer).**

Expected output:

```
+-----------+-------------------+-------+-------+
|customer_id|         event_time|   page|session|
+-----------+-------------------+-------+-------+
|          1|2024-03-01 09:00:00|   home|      1|
|          1|2024-03-01 09:10:00|product|      1|
|          1|2024-03-01 11:00:00|   home|      2|
|          1|2024-03-02 10:00:00|   cart|      3|
|          1|2024-03-03 10:30:00|   home|      4|
|          1|2024-03-05 08:00:00|   home|      5|
|          2|2024-03-01 20:00:00|   home|      1|
|          2|2024-03-01 20:20:00|product|      1|
|          2|2024-03-01 20:45:00|   cart|      1|
|          2|2024-03-04 18:00:00|   home|      2|
|          2|2024-03-05 18:00:00|   home|      3|
+-----------+-------------------+-------+-------+
```

<details><summary>Hint</summary>

Mark the first event of each session with 1 (others 0), then a running sum of the marks numbers the sessions.

</details>

**B8. For every customer, count the sessions (same 30-minute rule) and the length of the longest session in minutes. Show customer_id, sessions and longest_minutes.**

Expected output:

```
+-----------+--------+---------------+
|customer_id|sessions|longest_minutes|
+-----------+--------+---------------+
|          1|       5|             10|
|          2|       3|             45|
|          3|       3|              0|
+-----------+--------+---------------+
```

<details><summary>Hint</summary>

Number the sessions like in B7, then group by customer and session, then by customer.

</details>

**B9. What is each customer's longest streak of consecutive days with at least one event? Show customer_id and longest_streak.**

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             2|
|          3|             3|
+-----------+--------------+
```

<details><summary>Hint</summary>

Gaps and islands: number each customer's distinct days with `row_number` and subtract that number from the date. Days of one streak get the same result, so group by it.

</details>

**B10. Which days are missing between each customer's first and last event day (days without any event)? Show customer_id and missing_day.**

Expected output:

```
+-----------+-----------+
|customer_id|missing_day|
+-----------+-----------+
|          1| 2024-03-04|
|          2| 2024-03-02|
|          2| 2024-03-03|
+-----------+-----------+
```

<details><summary>Hint</summary>

Build every day from the first to the last with `F.sequence` and `F.explode`, then keep the days with no event (`left_anti`).

</details>

**B11. Which customers ordered more items in every order than in their previous one (at least 2 orders)? Show customer_id.**

Expected output:

```
+-----------+
|customer_id|
+-----------+
|          5|
+-----------+
```

<details><summary>Hint</summary>

Compare each quantity with the previous one (`F.lag`), count the orders that were not bigger, and keep the customers where that count is 0.

</details>

**B12. For each customer, show the first and the last page of their first session (30-minute rule, duplicates removed). Columns: customer_id, first_page and last_page.**

Expected output:

```
+-----------+----------+---------+
|customer_id|first_page|last_page|
+-----------+----------+---------+
|          1|      home|  product|
|          2|      home|     cart|
|          3|   product|  product|
+-----------+----------+---------+
```

<details><summary>Hint</summary>

Number the sessions like in B7 and keep session 1; `F.min_by(page, event_time)` is the page at the earliest time.

</details>

---
**Done?** In the Theory part of Day 6, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Which customer has the most orders? Then run the plain join of skewed_orders with many_customers and look at ...</summary>

**Customer 1 has 60% of the orders, and the biggest join task reads about 25 times more rows than the median task (and takes several times longer).** That is **data skew**: the data is unevenly spread over the keys, so after a shuffle by the key one partition is huge. All rows of customer 1 must go to the same task, which becomes a straggler: the stage, and every stage after it, waits for that one task while the other cores sit idle. In the Spark UI you see it in the stage's summary metrics: max far above the median for duration and shuffle read size or records, and one long bar in the event timeline. Skew is dangerous because one task can run out of memory or spill heavily, and adding executors doesn't help: one task runs on one core.

```python
skewed_orders.groupBy("customer_id").count().orderBy(F.desc("count")).show(3)
plain_join = skewed_orders.join(many_customers, "customer_id")
noop(plain_join)
print(task_report())
```

Result:

```
+-----------+-------+
|customer_id|  count|
+-----------+-------+
|          1|2399334|
|      59652|     37|
|      71365|     35|
+-----------+-------+
only showing top 3 rows
16 tasks | rows per task: median 106,193, max 2,506,812 | time: median 124 ms, max 495 ms
```

*Your times will differ; the row counts will not.*

</details>

<details>
<summary><b>A2</b> &nbsp;Fix it with salting: give every order a salt from 0 to 15, copy every customer 16 times (once per salt), and ...</summary>

**The biggest task now reads about 650,000 rows instead of 2.5 million, close to the median.** The salt splits customer 1's orders into 16 groups that land in different partitions; the customer side is copied 16 times so every group still finds its match. The price: the small side grows 16 times (1.6 million rows here), so the whole job can get slower even though it is balanced.

```python
SALTS = 16
salted_orders = skewed_orders.withColumn("salt", F.pmod(F.xxhash64("order_id"), F.lit(SALTS)))      # 0..15 per order
salted_customers = many_customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(SALTS - 1))))   # 16 copies
salted_join = salted_orders.join(salted_customers, ["customer_id", "salt"])
noop(salted_join)
print(task_report())
```

Result:

```
16 tasks | rows per task: median 349,950, max 652,222 | time: median 195 ms, max 592 ms
```

*Your times will differ; the row counts will not.*

</details>

<details>
<summary><b>A3</b> &nbsp;Salt only the hot key (a selective, or iterative, salting): customer 1 gets salts 0 to 15, everybody else ...</summary>

**100,016 customer rows instead of 1.6 million, and the tasks stay balanced.** Only the skewed key needs the copies. In practice you find the hot keys first (a `groupBy` count, as in A1), salt those, and check the task report again; if some keys are still too big, you salt them with more values: that repeated loop is what *iterative salting* means.

```python
hot = F.col("customer_id") == 1
hot_orders = skewed_orders.withColumn("salt", F.when(hot, F.pmod(F.xxhash64("order_id"), F.lit(SALTS))).otherwise(0))
hot_customers = (many_customers
                 .withColumn("salt", F.when(hot, F.sequence(F.lit(0), F.lit(SALTS - 1))).otherwise(F.array(F.lit(0))))
                 .withColumn("salt", F.explode("salt")))
print(hot_customers.count(), "customer rows instead of", many_customers.count() * SALTS)
noop(hot_orders.join(hot_customers, ["customer_id", "salt"]))
print(task_report())
```

Result:

```
100016 customer rows instead of 1600016
16 tasks | rows per task: median 255,664, max 557,810 | time: median 142 ms, max 192 ms
```

*Your times will differ; the row counts will not.*

</details>

<details>
<summary><b>A4</b> &nbsp;Let AQE handle it: turn AQE and its skew join on (cell below, with lower limits because our data is small) ...</summary>

**`SortMergeJoin(skew=true)`, with `AQEShuffleRead coalesced and skewed`.** After the shuffle, AQE measured the partitions, found one much bigger than the median (5 times and over the threshold), split it into several tasks, and copied the matching part of the other side for each of them. This works only for sort-merge and shuffle hash joins, and only when the partition sizes are known after a shuffle: it is the easiest fix, and the first one to try.

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "4MB")   # our data is small,
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "4MB")               # so lower the limits
by_region = skewed_orders.join(many_customers, "customer_id").groupBy("region").agg(F.sum("amount"))
by_region.collect()
by_region.explain()
```

Result:

```
AdaptiveSparkPlan isFinalPlan=true
      +- AQEShuffleRead coalesced
                     +- *(5) SortMergeJoin(skew=true) [customer_id#32L], [customer_id#35L], Inner
                        :  +- AQEShuffleRead coalesced and skewed
                           +- AQEShuffleRead coalesced
            +- SortMergeJoin [customer_id#32L], [customer_id#35L], Inner
```

</details>

<details>
<summary><b>A5</b> &nbsp;Two more fixes: broadcast the small side, or filter the hot key out if you don't need it (for example a test ...</summary>

**The broadcast join is the fastest of all (no shuffle, so no skewed partition), and without customer 1 the tasks are even.** Broadcasting works when the other side fits in memory. Filtering helps only when the hot rows are junk you may drop (test accounts, nulls, a default value like `unknown`); filtering early also shrinks every later shuffle. In short, the ways to fix skew: AQE skew join, broadcast the small side, salt the hot keys, filter or handle the hot keys separately, and choose better keys.

```python
start = time.time()
noop(skewed_orders.join(F.broadcast(many_customers), "customer_id"))
print(f"broadcast join: {time.time() - start:.2f} s")
without_reseller = skewed_orders.filter(F.col("customer_id") != 1)
noop(without_reseller.join(many_customers, "customer_id"))
print(task_report())
```

Result:

```
broadcast join: 0.35 s
8 tasks | rows per task: median 212,651, max 214,530 | time: median 76 ms, max 78 ms
```

*Your times will differ; the row counts will not.*

</details>

<details>
<summary><b>A6</b> &nbsp;What do repartition(8) and repartition(8, "customer_id") do to the partition sizes of skewed_orders? Then add ...</summary>

**`repartition(8)` gives 8 equal partitions; `repartition(8, "customer_id")` keeps the skew (one partition with all of customer 1's 2.4 million rows). `coalesce(2)` just merges neighbouring partitions: even stays even, skewed stays skewed.** `repartition(n)` is a full shuffle that spreads rows round-robin, so it fixes uneven partition sizes, but not a hot key once you repartition by that key. `coalesce(n)` only reduces the number of partitions, without a shuffle, by gluing existing ones together. Use `repartition` to increase partitions or to rebalance; use `coalesce` to reduce partitions cheaply, for example before writing fewer files.

```python
print("repartition(8):          ", partition_sizes(skewed_orders.repartition(8))[:3])
print("repartition(8, customer): ", partition_sizes(skewed_orders.repartition(8, "customer_id"))[:3])
print("repartition(8).coalesce(2):", partition_sizes(skewed_orders.repartition(8).coalesce(2)))
print("by customer, coalesce(2):  ", partition_sizes(skewed_orders.repartition(8, "customer_id").coalesce(2)))
```

Result:

```
repartition(8):           [500000, 500000, 500000]
repartition(8, customer):  [2599576, 201972, 201287]
repartition(8).coalesce(2): [2000000, 2000000]
by customer, coalesce(2):   [3202571, 797429]
```

</details>

<details>
<summary><b>A7</b> &nbsp;Can skew happen when reading? Write skewed_orders as one gzip CSV file and read it back: how many read ...</summary>

**1 read partition: one task reads all 4 million rows.** A gzip file can't be split (Lab 04), and the same happens with one huge file in a non-splittable format, or with a few big files next to many tiny ones. Fix the stages after the read with `repartition(n)` (one shuffle, then even partitions), and fix the source by writing splittable files of similar size.

```python
skewed_orders.coalesce(1).write.mode("overwrite").option("header", True).option("compression", "gzip").csv("data/shop/orders_gz")
gz = spark.read.option("header", True).csv("data/shop/orders_gz")
print(gz.rdd.getNumPartitions(), "read partition(s)")
print("after repartition(8):", partition_sizes(gz.repartition(8))[:3])
```

Result:

```
1 read partition(s)
after repartition(8): [500000, 500000, 500000]
```

</details>

<details>
<summary><b>B1</b> &nbsp;Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("rn", F.row_number().over(w))
       .filter(F.col("rn") == 1)
       .select("customer_id", "order_id", "order_date", "status")
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B2</b> &nbsp;Show each customer's second most recent order: customer_id, order_id and order_date. Customers with only one ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("rn", F.row_number().over(w))
       .filter(F.col("rn") == 2)
       .select("customer_id", "order_id", "order_date")
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;Which events are exact duplicates? Show customer_id, event_time, page and copies (how many times the row ...</summary>

```python
(events.groupBy("customer_id", "event_time", "page")
        .count()
        .withColumnRenamed("count", "copies")
        .filter(F.col("copies") > 1)
        .orderBy("customer_id")
        .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;Remove the exact duplicates, then count the events of each customer. Show customer_id and events.</summary>

```python
(events.dropDuplicates()
        .groupBy("customer_id").count()
        .withColumnRenamed("count", "events")
        .orderBy("customer_id")
        .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;Show the first visit of each page by each customer: customer_id, page and first_visit, sorted by customer_id, ...</summary>

```python
(events.groupBy("customer_id", "page")
        .agg(F.min("event_time").alias("first_visit"))
        .orderBy("customer_id", "first_visit")
        .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;For customer 1 (duplicates removed), show each event_time and the minutes since the previous event ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("event_time")
(events.dropDuplicates()
       .filter(F.col("customer_id") == 1)
       .withColumn("minutes_since_prev",
                   ((F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60).cast("int"))
       .select("event_time", "minutes_since_prev")
       .orderBy("event_time")
       .show())
```

</details>

<details>
<summary><b>B7</b> &nbsp;Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("event_time")
gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
(events.dropDuplicates()
       .filter(F.col("customer_id").isin(1, 2))
       .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
       .withColumn("session", F.sum("new_session").over(w))
       .select("customer_id", "event_time", "page", "session")
       .orderBy("customer_id", "event_time")
       .show())
```

</details>

<details>
<summary><b>B8</b> &nbsp;For every customer, count the sessions (same 30-minute rule) and the length of the longest session in ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("event_time")
gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
sessions = (events.dropDuplicates()
                  .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                  .withColumn("session", F.sum("new_session").over(w))
                  .groupBy("customer_id", "session")
                  .agg(((F.unix_timestamp(F.max("event_time")) - F.unix_timestamp(F.min("event_time"))) / 60)
                       .cast("int").alias("minutes")))
(sessions.groupBy("customer_id")
         .agg(F.count("*").alias("sessions"), F.max("minutes").alias("longest_minutes"))
         .orderBy("customer_id")
         .show())
```

</details>

<details>
<summary><b>B9</b> &nbsp;What is each customer's longest streak of consecutive days with at least one event? Show customer_id and ...</summary>

```python
days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
w = Window.partitionBy("customer_id").orderBy("day")
streaks = (days.withColumn("group", F.date_sub("day", F.row_number().over(w)))
               .groupBy("customer_id", "group").count())
(streaks.groupBy("customer_id")
        .agg(F.max("count").alias("longest_streak"))
        .orderBy("customer_id")
        .show())
```

</details>

<details>
<summary><b>B10</b> &nbsp;Which days are missing between each customer's first and last event day (days without any event)? Show ...</summary>

```python
days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
all_days = (days.groupBy("customer_id")
                .agg(F.explode(F.sequence(F.min("day"), F.max("day"))).alias("day")))
(all_days.join(days, ["customer_id", "day"], "left_anti")
         .withColumnRenamed("day", "missing_day")
         .orderBy("customer_id", "missing_day")
         .show())
```

</details>

<details>
<summary><b>B11</b> &nbsp;Which customers ordered more items in every order than in their previous one (at least 2 orders)? Show ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("prev_quantity", F.lag("quantity").over(w))
       .groupBy("customer_id")
       .agg(F.count("*").alias("orders"),
            F.sum(F.when(F.col("quantity") <= F.col("prev_quantity"), 1).otherwise(0)).alias("not_bigger"))
       .filter((F.col("orders") >= 2) & (F.col("not_bigger") == 0))
       .select("customer_id")
       .show())
```

</details>

<details>
<summary><b>B12</b> &nbsp;For each customer, show the first and the last page of their first session (30-minute rule, duplicates ...</summary>

```python
w = Window.partitionBy("customer_id").orderBy("event_time")
gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
first_session = (events.dropDuplicates()
                       .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                       .withColumn("session", F.sum("new_session").over(w))
                       .filter(F.col("session") == 1))
(first_session.groupBy("customer_id")
              .agg(F.min_by("page", "event_time").alias("first_page"), F.max_by("page", "event_time").alias("last_page"))
              .orderBy("customer_id")
              .show())
```

</details>